# Private Groq live verification

1. Open this notebook in Google Colab. Upload the matching `ai-platform-architect-labs.zip` using Colab's Files panel.
2. In the left **Secrets** panel (key icon), create `GROQ_API_KEY`, paste the key there and enable notebook access. Do not paste it into a code cell or this conversation.
3. Set `RUN_LIVE=True` below and run all cells. The example model is listed in Groq's model documentation as of 4 October 2026; your account must have access. You may change the exact model ID.
4. Download `groq-live-report.json` at the end and share that report. It contains fictional task inputs, actual answers and telemetry; it excludes the credential. Review it before sharing.

This performs up to 30 model requests, with no automatic retry. Charges/rate limits follow your account. It checks six typed requests, native tool use and nine policy cases. Human semantic review and embeddings are separate; this report does not auto-certify production readiness.

Official references: [Groq models](https://console.groq.com/docs/models), [local tool calling](https://console.groq.com/docs/tool-use/local-tool-calling), [Google's Colab Secrets example](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb).

In [ ]:
from pathlib import Path
import sys,zipfile,os
RUN_LIVE=False
MODEL='openai/gpt-oss-20b'
# Locate an extracted package, or extract the uploaded course ZIP in Colab.
candidates=[Path.cwd(),Path.cwd().parent,Path('/content/ai-platform-architect-labs')]
ROOT=next((p for p in candidates if (p/'engine').is_dir()),None)
if ROOT is None and Path('/content/ai-platform-architect-labs.zip').exists():
    with zipfile.ZipFile('/content/ai-platform-architect-labs.zip') as z:z.extractall('/content')
    ROOT=Path('/content/ai-platform-architect-labs')
if ROOT is None:raise RuntimeError('Upload the matching student ZIP to Colab Files, or run inside its extracted directory.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
print('Course files located. Live execution:',RUN_LIVE)

In [ ]:
if RUN_LIVE:
    try:
        from google.colab import userdata
    except ImportError:
        import getpass
        key=getpass.getpass('Groq key (hidden; not saved): ')
    else:
        key=userdata.get('GROQ_API_KEY')
    from verification.groq_check import run
    report_path,summary=run(key,MODEL)
    del key
    print(summary)
    print('Report:',report_path)
else:
    print('No API call made. Set RUN_LIVE=True after configuring the secret.')

In [ ]:
if RUN_LIVE:
    try:
        from google.colab import files
    except ImportError:
        print('Open the report in outputs/ and share it after review.')
    else:
        files.download(str(report_path))

## Interpreting the result

A failed extraction is evidence to investigate, not a reason to change the expected answer silently. A native tool pass means the correct authoritative tool total was observed; inspect the final prose separately. Policy answers have `human_correct`, `human_supported` and `human_complete` left unset for review. Provider failures are recorded without printing credentials or raw HTTP error bodies.

This notebook does not run an embedding endpoint. The main preflight/data-plane notebook covers the separately configured embedding service. Saving a secret in your Colab account does not grant this chat access to it; sharing the result file is the intended workflow.